# Fylla Public Workflow

This public notebook summarizes the publishable workflow behind the periocular-analysis extension I designed for Fylla.

This version is intentionally sanitized for public release:

- the original Colab notebook has been reduced to a portfolio-safe workflow notebook,
- company-owned training assets and deployment glue are omitted,
- private datasets, checkpoints, and website integrations are not shipped here,
- the notebook is meant to document structure, not reproduce the internal pipeline.

Live product context: [deltaskincare.it](https://deltaskincare.it/)


## Workflow scope

The original internal notebook covered five main stages:

1. landmark-guided eye cropping,
2. label preparation and dataset-quality passes,
3. multi-task training for bags and age-related cues,
4. replay-based extension for dark-circle prediction,
5. checkpoint packaging for website inference.

This public version keeps the same structure while replacing private pieces with lightweight, shareable code.


## High-level pipeline

<table>
  <tr>
    <th align="center" width="25%">Region focus</th>
    <th align="center" width="25%">Backbone adaptation</th>
    <th align="center" width="25%">Label-quality pass</th>
    <th align="center" width="25%">Dark-circle extension</th>
  </tr>
  <tr>
    <td align="center">Detect the face, localize landmarks, and crop a stable under-eye region.</td>
    <td align="center">Start from a pretrained facial backbone and specialize it for periocular cues.</td>
    <td align="center">Filter likely noisy labels before relying on the training split as-is.</td>
    <td align="center">Add a dedicated dark-circle output on top of the earlier eye-area workflow.</td>
  </tr>
  <tr>
    <th align="center" width="25%">Retention strategy</th>
    <th align="center" width="25%">Threshold tuning</th>
    <th align="center" width="25%">Checkpoint export</th>
    <th align="center" width="25%">Website integration</th>
  </tr>
  <tr>
    <td align="center">Replay earlier periocular samples so the new task does not erase previous behavior.</td>
    <td align="center">Tune decision thresholds on validation behavior instead of relying only on raw logits.</td>
    <td align="center">Package the model into an inference checkpoint suitable for product use.</td>
    <td align="center">Expose the module through the Fylla website without releasing private code or weights.</td>
  </tr>
</table>


In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

PUBLIC_DATA_ROOT = Path('/path/to/public_safe_data')
CHECKPOINT_DIR = Path('/path/to/checkpoints')
PRIMARY_TASKS = ['bags_under_eyes', 'young_appearance']
EXTENDED_TASKS = PRIMARY_TASKS + ['dark_circles']
IMAGE_SIZE = 224


## Dataset framing

The periocular branch was built around a staged workflow.

- An initial phase focused on under-eye bags and age-related cues.
- A later phase extended the model toward dark-circle analysis.
- The public notebook does not ship the original datasets or the exact data organization used in the company workflow.

What matters here is the workflow design: keep the labels explicit, keep the eye region stable, and treat later task additions as extensions of a deployed pipeline rather than isolated retraining runs.


In [ ]:
PUBLIC_LABEL_COLUMNS = {
    'Bags_Under_Eyes': 'bags_under_eyes',
    'Young': 'young_appearance',
    'Dark_Circles': 'dark_circles',
}


def build_publishable_label_table(attributes: pd.DataFrame) -> pd.DataFrame:
    """Rename the public-safe target columns and map labels to {0, 1}."""
    available = {
        source_name: public_name
        for source_name, public_name in PUBLIC_LABEL_COLUMNS.items()
        if source_name in attributes.columns
    }
    if not available:
        raise ValueError('Expected at least one publishable label column.')

    label_table = attributes.loc[:, available.keys()].rename(columns=available).copy()
    return label_table.replace({-1: 0, 1: 1})


def label_prevalence(label_table: pd.DataFrame) -> pd.Series:
    return label_table.mean(axis=0).sort_index()


## Region-focused preprocessing

A core design choice was to avoid full-face training for this branch. The workflow first localizes the eye landmarks and then crops a stable under-eye region before training or inference.

That keeps the model focused on the visual area that actually matters for periocular cues and makes later deployment more predictable.


In [ ]:
LEFT_EYE_POINTS = list(range(36, 42))
RIGHT_EYE_POINTS = list(range(42, 48))


def compute_under_eye_box(
    landmarks: np.ndarray,
    x_pad: float = 0.18,
    y_top_pad: float = 0.10,
    y_bottom_pad: float = 0.40,
) -> dict[str, int]:
    """Return a stable crop box around the periocular area."""
    points = np.asarray(landmarks, dtype=float)
    eye_points = np.concatenate([points[LEFT_EYE_POINTS], points[RIGHT_EYE_POINTS]], axis=0)

    x_min, y_min = eye_points.min(axis=0)
    x_max, y_max = eye_points.max(axis=0)
    width = max(x_max - x_min, 1.0)
    height = max(y_max - y_min, 1.0)

    return {
        'x0': int(max(0, x_min - width * x_pad)),
        'y0': int(max(0, y_min - height * y_top_pad)),
        'x1': int(x_max + width * x_pad),
        'y1': int(y_max + height * y_bottom_pad),
    }


## Quality filtering and replay-based extension

The internal Colab workflow included a label-quality pass before training and later reused part of the earlier eye-area data when the dark-circle task was introduced.

That replay step mattered because the real product problem was not just to add a new output, but to add it without erasing useful behavior that the earlier periocular branch had already learned.


In [ ]:
def select_clean_indices(issue_mask: np.ndarray) -> np.ndarray:
    """Keep only the rows that pass a quality-screening step."""
    issue_mask = np.asarray(issue_mask, dtype=bool)
    return np.flatnonzero(~issue_mask)


def build_replay_manifest(
    previous_stage_df: pd.DataFrame,
    new_task_df: pd.DataFrame,
    replay_fraction: float = 0.35,
    seed: int = 7,
) -> pd.DataFrame:
    """Mix prior-task samples with new-task samples for the extension stage."""
    replay_count = max(1, int(len(previous_stage_df) * replay_fraction))
    replay_rows = previous_stage_df.sample(
        n=min(replay_count, len(previous_stage_df)),
        random_state=seed,
    ).assign(training_stage='replay')

    new_rows = new_task_df.copy().assign(training_stage='new_task')
    return pd.concat([replay_rows, new_rows], ignore_index=True)


## Multi-task model design

The first public-safe stage can be summarized as a pretrained visual backbone plus a lightweight task head. The original training notebook contains the real backbone loading, dataloaders, checkpoints, and experiment tracking; the public notebook keeps only the shareable structure.


In [ ]:
class PublicPeriocularHead(nn.Module):
    def __init__(self, in_features: int, num_outputs: int):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_outputs),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.classifier(features)


TRAINING_CONFIG = {
    'image_size': IMAGE_SIZE,
    'batch_size': 32,
    'learning_rate': 5e-5,
    'loss': 'BCEWithLogitsLoss',
    'notes': 'The private notebook includes full dataloaders, checkpoints, and experiment tracking.',
}


def build_loss_weights(label_table: pd.DataFrame) -> torch.Tensor:
    prevalence = label_table.mean(axis=0).clip(lower=1e-6, upper=1 - 1e-6)
    return torch.tensor(((1.0 - prevalence) / prevalence).values, dtype=torch.float32)


## Deployment handoff

The final stage of the internal workflow was not just model training. The resulting checkpoint had to be packaged with task metadata and decision thresholds so the Fylla website could call the periocular branch consistently during analysis.

The product integration code remains private, but the handoff contract can still be shown safely.


In [ ]:
@dataclass
class ExportBundle:
    checkpoint_name: str
    tasks: list[str]
    thresholds: dict[str, float]


def build_export_bundle(checkpoint_name: str, thresholds: dict[str, float]) -> ExportBundle:
    return ExportBundle(
        checkpoint_name=checkpoint_name,
        tasks=EXTENDED_TASKS,
        thresholds=thresholds,
    )


# Example public-safe execution sketch
# attrs = pd.read_csv(PUBLIC_DATA_ROOT / 'labels.csv')
# label_table = build_publishable_label_table(attrs)
# clean_indices = select_clean_indices(issue_mask=np.zeros(len(label_table), dtype=bool))
# loss_weights = build_loss_weights(label_table.iloc[clean_indices][PRIMARY_TASKS])
# export_bundle = build_export_bundle(
#     checkpoint_name='fylla_periocular_public.ckpt',
#     thresholds={'bags_under_eyes': 0.48, 'young_appearance': 0.50, 'dark_circles': 0.44},
# )
# export_bundle
